In [1]:
import os

# Cria as pastas do projeto no ambiente do Colab
os.makedirs("models", exist_ok=True)
os.makedirs("core", exist_ok=True)
os.makedirs("services", exist_ok=True)
os.makedirs("ui", exist_ok=True)
os.makedirs("data", exist_ok=True)

print("Estrutura de pastas criada com sucesso!")

Estrutura de pastas criada com sucesso!


In [2]:
%%writefile models/client.py
class Client:
    def __init__(self, client_id: int, name: str, phone: str, is_priority: bool):
        self.id = client_id
        self.name = name
        self.phone = phone
        self.is_priority = is_priority
        self.is_active = True

    def __str__(self):
        status = "Prioritário" if self.is_priority else "Comum"
        return f"[{self.id}] {self.name} ({status})"

Writing models/client.py


In [3]:
%%writefile models/attendant.py
class Attendant:
    def __init__(self, attendant_id: int, name: str):
        self.id = attendant_id
        self.name = name
        self.is_busy = False

Writing models/attendant.py


In [4]:
%%writefile models/attendance.py
from datetime import datetime

class Attendance:
    def __init__(self, client_id: int, client_name: str, attendant_name: str, duration: int, date_str: str = None):
        self.client_id = client_id
        self.client_name = client_name
        self.attendant_name = attendant_name
        self.duration = duration  # em minutos
        self.date = date_str if date_str else datetime.now().strftime("%Y-%m-%d %H:%M")

    def to_csv(self) -> str:
        return f"{self.client_id},{self.client_name},{self.attendant_name},{self.duration},{self.date}"

Writing models/attendance.py


In [5]:
%%writefile core/structures.py
from typing import List, Optional
from models.client import Client

class Node:
    def __init__(self, client: Client):
        self.client = client
        self.next: Optional['Node'] = None

class LinkedList:
    """Lista Encadeada para gerenciamento de clientes ativos/inativos."""
    def __init__(self):
        self.head: Optional[Node] = None

    def append(self, client: Client):
        new_node = Node(client)
        if not self.head:
            self.head = new_node
            return
        current = self.head
        while current.next:
            current = current.next
        current.next = new_node

    def remove_inactive(self) -> int:
        """Remove nós onde o cliente está marcado como inativo."""
        count = 0
        while self.head and not self.head.client.is_active:
            self.head = self.head.next
            count += 1

        current = self.head
        while current and current.next:
            if not current.next.client.is_active:
                current.next = current.next.next
                count += 1
            else:
                current = current.next
        return count

class Queue:
    """Fila Comum baseada em lista nativa do Python."""
    def __init__(self):
        self.items: List[Client] = []

    def enqueue(self, item: Client):
        self.items.append(item)

    def dequeue(self) -> Optional[Client]:
        return self.items.pop(0) if not self.is_empty() else None

    def is_empty(self) -> bool:
        return len(self.items) == 0

class Stack:
    """Pilha para o mecanismo de Desfazer (Undo)."""
    def __init__(self):
        self.items = []

    def push(self, item):
        self.items.append(item)

    def pop(self):
        return self.items.pop() if not self.is_empty() else None

    def is_empty(self) -> bool:
        return len(self.items) == 0

Writing core/structures.py


In [6]:
%%writefile core/algorithms.py
from typing import List
from models.client import Client

def recursive_binary_search(arr: List[Client], low: int, high: int, target_id: int) -> int:
    """Busca Binária Recursiva para encontrar o índice do cliente pelo ID."""
    if low > high:
        return -1

    mid = (low + high) // 2
    if arr[mid].id == target_id:
        return mid
    elif arr[mid].id > target_id:
        return recursive_binary_search(arr, low, mid - 1, target_id)
    else:
        return recursive_binary_search(arr, mid + 1, high, target_id)

def merge_sort_attendances(arr: list) -> list:
    """Merge Sort para ordenação de relatórios por duração de atendimento."""
    if len(arr) <= 1:
        return arr

    mid = len(arr) // 2
    left = merge_sort_attendances(arr[:mid])
    right = merge_sort_attendances(arr[mid:])

    return merge(left, right)

def merge(left: list, right: list) -> list:
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i].duration >= right[j].duration:  # Ordem decrescente
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    result.extend(left[i:])
    result.extend(right[j:])
    return result

Writing core/algorithms.py


In [7]:
%%writefile services/manager.py
import os
from typing import List, Optional
from models.client import Client
from models.attendant import Attendant
from models.attendance import Attendance
from core.structures import Queue, Stack, LinkedList
from core.algorithms import recursive_binary_search, merge_sort_attendances

class MedTechManager:
    def __init__(self):
        self.clients_vector: List[Client] = []  # Vetor ordenado para busca
        self.temp_vector: List[Client] = []     # Vetor não ordenado
        self.active_clients_list = LinkedList()

        self.common_queue = Queue()
        self.priority_queue = Queue()
        self.undo_stack = Stack()

        self.attendants: List[Attendant] = []
        self.history: List[Attendance] = []

        self.load_data()

    def register_client(self, client_id: int, name: str, phone: str, is_priority: bool) -> bool:
        if self.search_client_by_id(client_id) is not None:
            return False  # ID já cadastrado

        new_client = Client(client_id, name, phone, is_priority)
        self.temp_vector.append(new_client)

        # Insere mantendo a ordenação para a busca binária
        self.clients_vector.append(new_client)
        self.clients_vector.sort(key=lambda c: c.id)

        self.active_clients_list.append(new_client)
        return True

    def register_attendant(self, attendant_id: int, name: str):
        self.attendants.append(Attendant(attendant_id, name))

    def search_client_by_id(self, client_id: int) -> Optional[Client]:
        idx = recursive_binary_search(self.clients_vector, 0, len(self.clients_vector) - 1, client_id)
        return self.clients_vector[idx] if idx != -1 else None

    def enter_queue(self, client_id: int) -> bool:
        client = self.search_client_by_id(client_id)
        if not client or not client.is_active:
            return False

        if client.is_priority:
            self.priority_queue.enqueue(client)
        else:
            self.common_queue.enqueue(client)
        return True

    def call_next(self, attendant_id: int) -> Optional[str]:
        attendant = next((a for a in self.attendants if a.id == attendant_id), None)
        if not attendant or attendant.is_busy:
            return None

        # Regra de Negócio: Prioridade sempre passa na frente
        if not self.priority_queue.is_empty():
            client = self.priority_queue.dequeue()
        elif not self.common_queue.is_empty():
            client = self.common_queue.dequeue()
        else:
            return None

        attendant.is_busy = True
        return f"Atendente {attendant.name} chamou Cliente {client.name} (ID: {client.id})"

    def finish_attendance(self, client_id: int, attendant_name: str, duration: int):
        client = self.search_client_by_id(client_id)
        if not client:
            return

        attendance = Attendance(client.id, client.name, attendant_name, duration)
        self.history.append(attendance)
        self.undo_stack.push(attendance)

        # Libera o atendente fictício
        for att in self.attendants:
            if att.name == attendant_name:
                att.is_busy = False

    def undo_last_attendance(self) -> bool:
        last = self.undo_stack.pop()
        if not last:
            return False
        self.history.remove(last)
        return True

    def remove_inactive_clients(self) -> int:
        return self.active_clients_list.remove_inactive()

    def get_average_time(self) -> float:
        if not self.history:
            return 0.0
        total_time = sum(att.duration for att in self.history)
        return total_time / len(self.history)

    def get_sorted_attendance_report(self) -> list:
        return merge_sort_attendances(self.history.copy())

    def export_to_csv(self, filename: str = "data/report_export.csv"):
        os.makedirs(os.path.dirname(filename), exist_ok=True)
        with open(filename, "w", encoding="utf-8") as f:
            f.write("id_cliente,nome_cliente,atendente,duracao,data\n")
            for att in self.history:
                f.write(att.to_csv() + "\n")

    def load_data(self):
        self.register_attendant(1, "Dr. Silva")
        self.register_attendant(2, "Dra. Maria")
        self.register_client(101, "Carlos Eduardo", "11999999999", False)
        self.register_client(105, "Ana Beatriz", "11888888888", True)
        self.register_client(102, "Bruno Souza", "11777777777", False)

Writing services/manager.py


In [8]:
%%writefile ui/terminal.py
import sys
from services.manager import MedTechManager

class TerminalUI:
    def __init__(self):
        self.manager = MedTechManager()

    def show_menu(self):
        print("\n" + "="*45)
        print("     MEDTECH SOLUTIONS - COUTING SYSTEM")
        print("="*45)
        print("1. Cadastrar Cliente")
        print("2. Inserir Cliente na Fila")
        print("3. Chamar Próximo da Fila")
        print("4. Finalizar Atendimento")
        print("5. Desfazer Última Finalização (Undo)")
        print("6. Buscar Cliente por ID")
        print("7. Relatório de Tempo Médio")
        print("8. Listar Atendimentos Ordenados (Merge Sort)")
        print("9. Expurgar Clientes Inativos da Lista")
        print("10. Exportar Dados para CSV")
        print("0. Sair")
        print("="*45)

    def run(self):
        while True:
            self.show_menu()
            try:
                option = input("Escolha uma opção: ").strip()
                if option == "1":
                    c_id = int(input("ID do Cliente: "))
                    name = input("Nome: ")
                    phone = input("Telefone: ")
                    pri = input("Prioritário? (s/n): ").lower() == 's'
                    if self.manager.register_client(c_id, name, phone, pri):
                        print("▶ Cliente cadastrado com sucesso!")
                    else:
                        print("❌ Erro: ID já existente.")

                elif option == "2":
                    c_id = int(input("ID do Cliente para a fila: "))
                    if self.manager.enter_queue(c_id):
                        print("▶ Cliente entrou na fila com sucesso!")
                    else:
                        print("❌ Cliente não localizado ou inativo.")

                elif option == "3":
                    att_id = int(input("ID do Atendente que vai chamar: "))
                    res = self.manager.call_next(att_id)
                    if res:
                        print(f"🔔 {res}")
                    else:
                        print("❌ Nenhum cliente na fila ou atendente ocupado.")

                elif option == "4":
                    c_id = int(input("ID do Cliente atendido: "))
                    att_name = input("Nome do Atendente: ")
                    duration = int(input("Duração em minutos: "))
                    self.manager.finish_attendance(c_id, att_name, duration)
                    print("▶ Atendimento finalizado e registrado.")

                elif option == "5":
                    if self.manager.undo_last_attendance():
                        print("🔄 Último atendimento desfeito com sucesso!")
                    else:
                        print("❌ Pilha de histórico vazia.")

                elif option == "6":
                    c_id = int(input("Digite o ID para busca rápida: "))
                    client = self.manager.search_client_by_id(c_id)
                    if client:
                        print(f"🔍 Encontrado: {client} | Tel: {client.phone}")
                    else:
                        print("❌ Cliente não encontrado.")

                elif option == "7":
                    avg = self.manager.get_average_time()
                    print(f"📊 Tempo Médio de Atendimento: {avg:.2f} minutos.")

                elif option == "8":
                    ordered = self.manager.get_sorted_attendance_report()
                    print("\n--- Relatório de Atendimentos (Mais Longos Primeiro) ---")
                    for att in ordered:
                        print(f"Cliente: {att.client_name} | Atendente: {att.attendant_name} | Duração: {att.duration} min")

                elif option == "9":
                    removed = self.manager.remove_inactive_clients()
                    print(f"♻️ Limpeza concluída. {removed} clientes removidos da lista encadeada.")

                elif option == "10":
                    self.manager.export_to_csv()
                    print("💾 Dados exportados com sucesso para 'data/report_export.csv'!")

                elif option == "0":
                    print("Encerrando MedTech Solutions. Até logo!")
                    break
                else:
                    print("⚠️ Opção Inválida!")
            except ValueError:
                print("❌ Erro de Entrada: Por favor, digite números válidos para IDs e tempos.")
            except Exception as e:
                print(f"💥 Ocorreu um erro inesperado: {e}")

Writing ui/terminal.py


In [9]:
%%writefile main.py
from ui.terminal import TerminalUI

if __name__ == "__main__":
    app = TerminalUI()
    app.run()

Writing main.py


In [10]:
!python main.py


     MEDTECH SOLUTIONS - COUTING SYSTEM
1. Cadastrar Cliente
2. Inserir Cliente na Fila
3. Chamar Próximo da Fila
4. Finalizar Atendimento
5. Desfazer Última Finalização (Undo)
6. Buscar Cliente por ID
7. Relatório de Tempo Médio
8. Listar Atendimentos Ordenados (Merge Sort)
9. Expurgar Clientes Inativos da Lista
10. Exportar Dados para CSV
0. Sair
Escolha uma opção: 1
ID do Cliente: 100
Nome: Kaua barroso silva
Telefone: 11 132132131
Prioritário? (s/n): s
▶ Cliente cadastrado com sucesso!

     MEDTECH SOLUTIONS - COUTING SYSTEM
1. Cadastrar Cliente
2. Inserir Cliente na Fila
3. Chamar Próximo da Fila
4. Finalizar Atendimento
5. Desfazer Última Finalização (Undo)
6. Buscar Cliente por ID
7. Relatório de Tempo Médio
8. Listar Atendimentos Ordenados (Merge Sort)
9. Expurgar Clientes Inativos da Lista
10. Exportar Dados para CSV
0. Sair
Escolha uma opção: 2
ID do Cliente para a fila: 100
▶ Cliente entrou na fila com sucesso!

     MEDTECH SOLUTIONS - COUTING SYSTEM
1. Cadastrar Cliente
